## Market Data Pipeline
This pipeline notebook will go through several data pages that Prof. Damodaran maintains on [his website](https://pages.stern.nyu.edu/~adamodar/New_Home_Page/home.htm).  Since I can't guarantee that his formatting will be the same, it would be best to run this notebook one cell at a time and ensure that the extracted data matches the proper format.

Damodaran generally updates this data yearly, during the first few weeks of January.

Due to the long lived nature of this data, the inconsistency for when Damodaran will update his webpages, and the potential for formatting changes, it would be best that this pipeline be run manually every year to keep the marketData.json up to date.

The pipeline will edit the marketData.json file in place, so it will need to be added and commited to deploy to the website.

This pipeline was developed using Python 3.12.3.  This pipeline will install the necessary pip packages, so it would be best to use a .venv environment.  A requirements.txt file is included in this folder to better control the requirements.

In [1]:
!pip install requests beautifulsoup4 pandas openpyxl

In [2]:
import json
from io import BytesIO

import requests
from bs4 import BeautifulSoup
import pandas as pd

In [3]:
# create dummy starting data for market data
market_data = {
  "rfr": 12.34,
  "taxRate": 25,
  "industries": {
    "Choose an Industry": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Advertising": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Aerospace/Defense": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Air Transport": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Apparel": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Auto & Truck": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Auto Parts": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Bank (Money Center)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Banks (Regional)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Beverage (Alcoholic)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Beverage (Soft)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Broadcasting": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Brokerage & Investment Banking": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Building Materials": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Business & Consumer Services": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Cable TV": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Chemical (Basic)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Chemical (Diversified)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Chemical (Specialty)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Coal & Related Energy": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Computer Services": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Computers/Peripherals": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Construction Supplies": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Diversified": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Drugs (Biotechnology)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Drugs (Pharmaceutical)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Education": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Electrical Equipment": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Electronics (Consumer & Office)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Electronics (General)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Engineering/Construction": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Entertainment": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Environmental & Waste Services": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Farming/Agriculture": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Financial Svcs. (Non-bank & Insurance)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Food Processing": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Food Wholesalers": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Furn/Home Furnishings": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Green & Renewable Energy": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Healthcare Products": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Healthcare Support Services": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Heathcare Information and Technology": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Homebuilding": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Hospitals/Healthcare Facilities": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Hotel/Gaming": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Household Products": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Information Services": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Insurance (General)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Insurance (Life)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Insurance (Prop/Cas.)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Investments & Asset Management": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Machinery": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Metals & Mining": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Office Equipment & Services": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Oil/Gas (Integrated)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Oil/Gas (Production and Exploration)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Oil/Gas Distribution": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Oilfield Svcs/Equip.": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Packaging & Container": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Paper/Forest Products": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Power": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Precious Metals": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Publishing & Newspapers": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "R.E.I.T.": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Real Estate (Development)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Real Estate (General/Diversified)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Real Estate (Operations & Services)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Recreation": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Reinsurance": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Restaurant/Dining": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Retail (Automotive)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Retail (Building Supply)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Retail (Distributors)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Retail (General)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Retail (Grocery and Food)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Retail (REITs)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Retail (Special Lines)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Rubber& Tires": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Semiconductor": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Semiconductor Equip": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Shipbuilding & Marine": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Shoe": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Software (Entertainment)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Software (Internet)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Software (System & Application)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Steel": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Telecom (Wireless)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Telecom. Equipment": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Telecom. Services": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Tobacco": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Transportation": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Transportation (Railroads)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Trucking": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Utility (General)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    },
    "Utility (Water)": {
      "EV/Sales": 1.1234,
      "Cost of Capital": 12.34,
      "Beta": 12.34,
      "Revenue Growth Rate": 12.34,
      "Operating Margin": 12.34,
      "Sales Cap Ratio": 12.34,
      "ROIC": 12.34,
      "Tax Rate": 12.34
    }
  },
  "countries": {
    "Select a Country": {
      "ERP": 12.34
    },
    "Abu Dhabi": {
      "ERP": 12.34
    },
    "Albania": {
      "ERP": 12.34
    },
    "Algeria": {
      "ERP": 12.34
    },
    "Andorra (Principality of)": {
      "ERP": 12.34
    },
    "Angola": {
      "ERP": 12.34
    },
    "Anguilla": {
      "ERP": 12.34
    },
    "Antigua & Barbuda": {
      "ERP": 12.34
    },
    "Argentina": {
      "ERP": 12.34
    },
    "Armenia": {
      "ERP": 12.34
    },
    "Aruba": {
      "ERP": 12.34
    },
    "Australia": {
      "ERP": 12.34
    },
    "Austria": {
      "ERP": 12.34
    },
    "Azerbaijan": {
      "ERP": 12.34
    },
    "Bahamas": {
      "ERP": 12.34
    },
    "Bahrain": {
      "ERP": 12.34
    },
    "Bangladesh": {
      "ERP": 12.34
    },
    "Barbados": {
      "ERP": 12.34
    },
    "Belarus": {
      "ERP": 12.34
    },
    "Belgium": {
      "ERP": 12.34
    },
    "Belize": {
      "ERP": 12.34
    },
    "Benin": {
      "ERP": 12.34
    },
    "Bermuda": {
      "ERP": 12.34
    },
    "Bolivia": {
      "ERP": 12.34
    },
    "Bosnia and Herzegovina": {
      "ERP": 12.34
    },
    "Botswana": {
      "ERP": 12.34
    },
    "Brazil": {
      "ERP": 12.34
    },
    "British Virgin Islands": {
      "ERP": 12.34
    },
    "Brunei": {
      "ERP": 12.34
    },
    "Bulgaria": {
      "ERP": 12.34
    },
    "Burkina Faso": {
      "ERP": 12.34
    },
    "Cambodia": {
      "ERP": 12.34
    },
    "Cameroon": {
      "ERP": 12.34
    },
    "Canada": {
      "ERP": 12.34
    },
    "Cape Verde": {
      "ERP": 12.34
    },
    "Cayman Islands": {
      "ERP": 12.34
    },
    "Channel Islands": {
      "ERP": 12.34
    },
    "Chile": {
      "ERP": 12.34
    },
    "China": {
      "ERP": 12.34
    },
    "Colombia": {
      "ERP": 12.34
    },
    "Congo (Democratic Republic of)": {
      "ERP": 12.34
    },
    "Congo (Republic of)": {
      "ERP": 12.34
    },
    "Cook Islands": {
      "ERP": 12.34
    },
    "Costa Rica": {
      "ERP": 12.34
    },
    "Croatia": {
      "ERP": 12.34
    },
    "Cuba": {
      "ERP": 12.34
    },
    "Curaçao": {
      "ERP": 12.34
    },
    "Cyprus": {
      "ERP": 12.34
    },
    "Czech Republic": {
      "ERP": 12.34
    },
    "Denmark": {
      "ERP": 12.34
    },
    "Dominican Republic": {
      "ERP": 12.34
    },
    "Ecuador": {
      "ERP": 12.34
    },
    "Egypt": {
      "ERP": 12.34
    },
    "El Salvador": {
      "ERP": 12.34
    },
    "Estonia": {
      "ERP": 12.34
    },
    "Ethiopia": {
      "ERP": 12.34
    },
    "Falkland Islands": {
      "ERP": 12.34
    },
    "Fiji": {
      "ERP": 12.34
    },
    "Finland": {
      "ERP": 12.34
    },
    "France": {
      "ERP": 12.34
    },
    "French Guiana": {
      "ERP": 12.34
    },
    "Gabon": {
      "ERP": 12.34
    },
    "Gambia": {
      "ERP": 12.34
    },
    "Georgia": {
      "ERP": 12.34
    },
    "Germany": {
      "ERP": 12.34
    },
    "Ghana": {
      "ERP": 12.34
    },
    "Gibraltar": {
      "ERP": 12.34
    },
    "Greece": {
      "ERP": 12.34
    },
    "Greenland": {
      "ERP": 12.34
    },
    "Guatemala": {
      "ERP": 12.34
    },
    "Guernsey (States of)": {
      "ERP": 12.34
    },
    "Guinea": {
      "ERP": 12.34
    },
    "Guinea-Bissau": {
      "ERP": 12.34
    },
    "Guyana": {
      "ERP": 12.34
    },
    "Haiti": {
      "ERP": 12.34
    },
    "Honduras": {
      "ERP": 12.34
    },
    "Hong Kong": {
      "ERP": 12.34
    },
    "Hungary": {
      "ERP": 12.34
    },
    "Iceland": {
      "ERP": 12.34
    },
    "India": {
      "ERP": 12.34
    },
    "Indonesia": {
      "ERP": 12.34
    },
    "Iran": {
      "ERP": 12.34
    },
    "Iraq": {
      "ERP": 12.34
    },
    "Ireland": {
      "ERP": 12.34
    },
    "Isle of Man": {
      "ERP": 12.34
    },
    "Israel": {
      "ERP": 12.34
    },
    "Italy": {
      "ERP": 12.34
    },
    "Ivory Coast": {
      "ERP": 12.34
    },
    "Jamaica": {
      "ERP": 12.34
    },
    "Japan": {
      "ERP": 12.34
    },
    "Jersey (States of)": {
      "ERP": 12.34
    },
    "Jordan": {
      "ERP": 12.34
    },
    "Kazakhstan": {
      "ERP": 12.34
    },
    "Kenya": {
      "ERP": 12.34
    },
    "Korea, D.P.R.": {
      "ERP": 12.34
    },
    "Kuwait": {
      "ERP": 12.34
    },
    "Kyrgyzstan": {
      "ERP": 12.34
    },
    "Laos": {
      "ERP": 12.34
    },
    "Latvia": {
      "ERP": 12.34
    },
    "Lebanon": {
      "ERP": 12.34
    },
    "Liberia": {
      "ERP": 12.34
    },
    "Libya": {
      "ERP": 12.34
    },
    "Liechtenstein": {
      "ERP": 12.34
    },
    "Lithuania": {
      "ERP": 12.34
    },
    "Luxembourg": {
      "ERP": 12.34
    },
    "Macau": {
      "ERP": 12.34
    },
    "Macedonia": {
      "ERP": 12.34
    },
    "Madagascar": {
      "ERP": 12.34
    },
    "Malawi": {
      "ERP": 12.34
    },
    "Malaysia": {
      "ERP": 12.34
    },
    "Maldives": {
      "ERP": 12.34
    },
    "Mali": {
      "ERP": 12.34
    },
    "Malta": {
      "ERP": 12.34
    },
    "Martinique": {
      "ERP": 12.34
    },
    "Mauritius": {
      "ERP": 12.34
    },
    "Mexico": {
      "ERP": 12.34
    },
    "Moldova": {
      "ERP": 12.34
    },
    "Monaco": {
      "ERP": 12.34
    },
    "Mongolia": {
      "ERP": 12.34
    },
    "Montenegro": {
      "ERP": 12.34
    },
    "Montserrat": {
      "ERP": 12.34
    },
    "Morocco": {
      "ERP": 12.34
    },
    "Mozambique": {
      "ERP": 12.34
    },
    "Myanmar": {
      "ERP": 12.34
    },
    "Namibia": {
      "ERP": 12.34
    },
    "Nepal": {
      "ERP": 12.34
    },
    "Netherlands": {
      "ERP": 12.34
    },
    "Netherlands Antilles": {
      "ERP": 12.34
    },
    "New Zealand": {
      "ERP": 12.34
    },
    "Nicaragua": {
      "ERP": 12.34
    },
    "Niger": {
      "ERP": 12.34
    },
    "Nigeria": {
      "ERP": 12.34
    },
    "Norway": {
      "ERP": 12.34
    },
    "Oman": {
      "ERP": 12.34
    },
    "Pakistan": {
      "ERP": 12.34
    },
    "Palestinian Authority": {
      "ERP": 12.34
    },
    "Panama": {
      "ERP": 12.34
    },
    "Papua New Guinea": {
      "ERP": 12.34
    },
    "Paraguay": {
      "ERP": 12.34
    },
    "Peru": {
      "ERP": 12.34
    },
    "Philippines": {
      "ERP": 12.34
    },
    "Poland": {
      "ERP": 12.34
    },
    "Portugal": {
      "ERP": 12.34
    },
    "Qatar": {
      "ERP": 12.34
    },
    "Ras Al Khaimah (Emirate of)": {
      "ERP": 12.34
    },
    "Reunion": {
      "ERP": 12.34
    },
    "Romania": {
      "ERP": 12.34
    },
    "Russia": {
      "ERP": 12.34
    },
    "Rwanda": {
      "ERP": 12.34
    },
    "Saint Lucia": {
      "ERP": 12.34
    },
    "Saudi Arabia": {
      "ERP": 12.34
    },
    "Senegal": {
      "ERP": 12.34
    },
    "Serbia": {
      "ERP": 12.34
    },
    "Sharjah": {
      "ERP": 12.34
    },
    "Sierra Leone": {
      "ERP": 12.34
    },
    "Singapore": {
      "ERP": 12.34
    },
    "Slovakia": {
      "ERP": 12.34
    },
    "Slovenia": {
      "ERP": 12.34
    },
    "Solomon Islands": {
      "ERP": 12.34
    },
    "Somalia": {
      "ERP": 12.34
    },
    "South Africa": {
      "ERP": 12.34
    },
    "South Korea": {
      "ERP": 12.34
    },
    "Spain": {
      "ERP": 12.34
    },
    "Sri Lanka": {
      "ERP": 12.34
    },
    "St. Maarten": {
      "ERP": 12.34
    },
    "St. Vincent & the Grenadines": {
      "ERP": 12.34
    },
    "Sudan": {
      "ERP": 12.34
    },
    "Suriname": {
      "ERP": 12.34
    },
    "Swaziland": {
      "ERP": 12.34
    },
    "Sweden": {
      "ERP": 12.34
    },
    "Switzerland": {
      "ERP": 12.34
    },
    "Syria": {
      "ERP": 12.34
    },
    "Taiwan": {
      "ERP": 12.34
    },
    "Tajikistan": {
      "ERP": 12.34
    },
    "Tanzania": {
      "ERP": 12.34
    },
    "Thailand": {
      "ERP": 12.34
    },
    "Togo": {
      "ERP": 12.34
    },
    "Trinidad & Tobago": {
      "ERP": 12.34
    },
    "Tunisia": {
      "ERP": 12.34
    },
    "Turkey": {
      "ERP": 12.34
    },
    "Turks & Caicos Islands": {
      "ERP": 12.34
    },
    "Uganda": {
      "ERP": 12.34
    },
    "Ukraine": {
      "ERP": 12.34
    },
    "United Arab Emirates": {
      "ERP": 12.34
    },
    "United Kingdom": {
      "ERP": 12.34
    },
    "United States": {
      "ERP": 12.34
    },
    "Uruguay": {
      "ERP": 12.34
    },
    "Uzbekistan": {
      "ERP": 12.34
    },
    "Venezuela": {
      "ERP": 12.34
    },
    "Vietnam": {
      "ERP": 12.34
    },
    "Yemen": {
      "ERP": 12.34
    },
    "Zambia": {
      "ERP": 12.34
    },
    "Zimbabwe": {
      "ERP": 12.34
    }
  },
  "regions": {
    "Select a Region": {
      "ERP": 12.34
    },
    "Africa": {
      "ERP": 12.34
    },
    "Asia": {
      "ERP": 12.34
    },
    "Australia & New Zealand": {
      "ERP": 12.34
    },
    "Caribbean": {
      "ERP": 12.34
    },
    "Central and South America": {
      "ERP": 12.34
    },
    "Eastern Europe": {
      "ERP": 12.34
    },
    "Middle East": {
      "ERP": 12.34
    },
    "North America": {
      "ERP": 12.34
    },
    "Western Europe": {
      "ERP": 12.34
    },
    "EMEA": {
      "ERP": 12.34
    },
    "Rest of World": {
      "ERP": 12.34
    }
  },
  "credit_ratings": {
    "Select a Credit Rating": {
      "Spread": 12.34,
      "gt_safe": 100001,
      "gt_risk": 100001,
      "lt_safe": -100001,
      "lt_risk": -100001
    },
    "Aaa/AAA": {
      "Spread": 12.34,
      "gt_safe": 8.5,
      "gt_risk": 12.5,
      "lt_safe": 100000,
      "lt_risk": 100000
    },
    "Aa2/AA": {
      "Spread": 12.34,
      "gt_safe": 6.5,
      "gt_risk": 9.5,
      "lt_safe": 8.5,
      "lt_risk": 12.5
    },
    "A1/A+": {
      "Spread": 12.34,
      "gt_safe": 5.5,
      "gt_risk": 7.5,
      "lt_safe": 6.5,
      "lt_risk": 9.5
    },
    "A2/A": {
      "Spread": 12.34,
      "gt_safe": 4.25,
      "gt_risk": 6,
      "lt_safe": 5.5,
      "lt_risk": 7.5
    },
    "A3/A-": {
      "Spread": 12.34,
      "gt_safe": 3,
      "gt_risk": 4.5,
      "lt_safe": 4.25,
      "lt_risk": 6
    },
    "Baa2/BBB": {
      "Spread": 12.34,
      "gt_safe": 2.5,
      "gt_risk": 4,
      "lt_safe": 3,
      "lt_risk": 4.5
    },
    "Ba1/BB+": {
      "Spread": 12.34,
      "gt_safe": 2.25,
      "gt_risk": 3.5,
      "lt_safe": 2.5,
      "lt_risk": 4
    },
    "Ba2/BB": {
      "Spread": 12.34,
      "gt_safe": 2,
      "gt_risk": 3,
      "lt_safe": 2.25,
      "lt_risk": 3.5
    },
    "B1/B+": {
      "Spread": 12.34,
      "gt_safe": 1.75,
      "gt_risk": 3,
      "lt_safe": 2,
      "lt_risk": 3.5
    },
    "B2/B": {
      "Spread": 12.34,
      "gt_safe": 1.5,
      "gt_risk": 2,
      "lt_safe": 1.75,
      "lt_risk": 2.5
    },
    "B3/B-": {
      "Spread": 12.34,
      "gt_safe": 1.25,
      "gt_risk": 1.5,
      "lt_safe": 1.5,
      "lt_risk": 2
    },
    "C2/C": {
      "Spread": 12.34,
      "gt_safe": 0.2,
      "gt_risk": 0.5,
      "lt_safe": 0.65,
      "lt_risk": 0.8
    },
    "Ca2/CC": {
      "Spread": 12.34,
      "gt_safe": 0.65,
      "gt_risk": 0.8,
      "lt_safe": 0.8,
      "lt_risk": 1.25
    },
    "Caa/CCC": {
      "Spread": 12.34,
      "gt_safe": 0.8,
      "gt_risk": 1.25,
      "lt_safe": 1.25,
      "lt_risk": 1.5
    },
    "D2/D":{
      "Spread": 12.34,
      "gt_safe": -100001,
      "gt_risk": -100001,
      "lt_safe": 0.2,
      "lt_risk": 0.5
    }
  }
}

In [ ]:
ctryprem_url = "https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/ctryprem.html"

try:
    response = requests.get(ctryprem_url)
    response.raise_for_status()

    ctryprem_soup = BeautifulSoup(response.text, 'html.parser')

    all_tables = ctryprem_soup.find_all('table')

    ctryprem_table = next(table for table in all_tables if 'Abu Dhabi' in str(table))
    if not ctryprem_table:
        raise ValueError("Could not find the country premium table in the HTML content.")
    
except Exception as e:
    print(f"Error fetching or parsing country premium data: {e}")
    ctryprem_table = None

ctryprem_rows = ctryprem_table.find_all('tr')[1:]
ctryprem_data = {}

for row in ctryprem_rows:
    cols = row.find_all('td')
    country_name = ' '.join(cols[0].get_text(strip=True).split())
    try:
        equity_risk_premium = float(cols[4].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid ERP value for country {country_name}: {e}")
        equity_risk_premium = 12.34

    try:
        tax_rate = float(cols[5].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid corp tax rate for country {country_name}: {e}")
        tax_rate = 12.34

    ctryprem_data[country_name] = {
        "ERP": equity_risk_premium,
        "Tax Rate": tax_rate
    }

print(json.dumps(ctryprem_data, indent=2))


Invalid ERP value for country : could not convert string to float: ''
Invalid corp tax rate for country : could not convert string to float: ''
{
  "Abu Dhabi": {
    "ERP": 4.87,
    "Tax Rate": 9.0
  },
  "Albania": {
    "ERP": 8.89,
    "Tax Rate": 15.0
  },
  "Algeria": {
    "ERP": 10.06,
    "Tax Rate": 10.07
  },
  "Andorra (Principality of)": {
    "ERP": 6.3,
    "Tax Rate": 10.0
  },
  "Angola": {
    "ERP": 12.64,
    "Tax Rate": 25.0
  },
  "Argentina": {
    "ERP": 13.94,
    "Tax Rate": 35.0
  },
  "Armenia": {
    "ERP": 8.89,
    "Tax Rate": 18.0
  },
  "Aruba": {
    "ERP": 7.08,
    "Tax Rate": 22.0
  },
  "Australia": {
    "ERP": 4.23,
    "Tax Rate": 30.0
  },
  "Austria": {
    "ERP": 4.59,
    "Tax Rate": 23.0
  },
  "Azerbaijan": {
    "ERP": 7.08,
    "Tax Rate": 20.0
  },
  "Bahamas": {
    "ERP": 10.06,
    "Tax Rate": 0.0
  },
  "Bahrain": {
    "ERP": 11.35,
    "Tax Rate": 0.0
  },
  "Bangladesh": {
    "ERP": 11.35,
    "Tax Rate": 27.5
  },
  "Barbados"

In [5]:
for country in market_data['countries']:
    if country == 'Select a Country':
        continue
    if country in ctryprem_data:
        market_data['countries'][country]['ERP'] = ctryprem_data[country]['ERP']
    else:
        print(f"Country '{country}' not found in fetched data; defaulting to None.")
        market_data['countries'][country]['ERP'] = None

for country in ctryprem_data:
    if country not in market_data['countries']:
        print(f"Fetched country '{country}' not found in market data; skipping.")

market_data['countries']['Trinidad & Tobago']['ERP'] = \
    ctryprem_data['Trinidad and Tobago']['ERP']
print('Manually replaced Trinidad & Tobago, due to naming mismatch.')

market_data['countries']['Ivory Coast']['ERP'] = \
    ctryprem_data['Côte d\'Ivoire']['ERP']
print('Manually replaced Ivory Coast, due to naming mismatch.')

market_data['countries']['Curaçao']['ERP'] = \
    ctryprem_data['Curacao']['ERP']
print('Manually replaced Curaçao, due to naming mismatch.')

market_data['countries']['South Korea']['ERP'] = \
    ctryprem_data['Korea']['ERP']
print('Manually replaced South Korea, due to naming mismatch.')

market_data['countries']['Macau']['ERP'] = \
    ctryprem_data['Macao']['ERP']
print('Manually replaced Macau, due to naming mismatch.')

market_data['countries']['Turks & Caicos Islands']['ERP'] = \
    ctryprem_data['Turks and Caicos Islands']['ERP']
print('Manually replaced Turks & Caicos Islands, due to naming mismatch.')

market_data['countries']['Yemen']['ERP'] = \
    ctryprem_data['Yemen, Republic']['ERP']
print('Manually replaced Yemen, due to naming mismatch.')

print(json.dumps(market_data['countries'], indent=2))

market_data['taxRate'] = ctryprem_data['United States']['Tax Rate']

print('tax rate', json.dumps(market_data['taxRate'], indent=2))

Country 'Anguilla' not found in fetched data; defaulting to None.
Country 'Antigua & Barbuda' not found in fetched data; defaulting to None.
Country 'British Virgin Islands' not found in fetched data; defaulting to None.
Country 'Channel Islands' not found in fetched data; defaulting to None.
Country 'Curaçao' not found in fetched data; defaulting to None.
Country 'Falkland Islands' not found in fetched data; defaulting to None.
Country 'French Guiana' not found in fetched data; defaulting to None.
Country 'Gibraltar' not found in fetched data; defaulting to None.
Country 'Greenland' not found in fetched data; defaulting to None.
Country 'Ivory Coast' not found in fetched data; defaulting to None.
Country 'Macau' not found in fetched data; defaulting to None.
Country 'Martinique' not found in fetched data; defaulting to None.
Country 'Monaco' not found in fetched data; defaulting to None.
Country 'Netherlands Antilles' not found in fetched data; defaulting to None.
Country 'Palestinian

In [6]:
psdata_url = "https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/psdata.html"

try:
    response = requests.get(psdata_url)
    response.raise_for_status()

    psdata_soup = BeautifulSoup(response.text, 'html.parser')

    all_tables = psdata_soup.find_all('table')

    psdata_table = next(table for table in all_tables if 'Advertising' in str(table))
    if not psdata_table:
        raise ValueError("Could not find the revenue multiples table in the HTML content.")
    
except Exception as e:
    print(f"Error fetching or parsing revenue multiples data: {e}")
    psdata_table = None

psdata_rows = psdata_table.find_all('tr')[1:]
psdata_data = {}

for row in psdata_rows:
    cols = row.find_all('td')
    industry_name = ' '.join(cols[0].get_text(strip=True).split())
    try:
        ev_sales = float(cols[4].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid EV/Sales value for industry {industry_name}: {e}")
        ev_sales = None

    try:
        margin = float(cols[5].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid operating margin for industry {industry_name}: {e}")
        margin = None

    psdata_data[industry_name] = {
        "EV/Sales": ev_sales,
        "Operating Margin": margin
    }

print(json.dumps(psdata_data, indent=2))

Invalid EV/Sales value for industry Bank (Money Center): could not convert string to float: 'NA'
Invalid operating margin for industry Bank (Money Center): could not convert string to float: 'NA'
Invalid EV/Sales value for industry Banks (Regional): could not convert string to float: 'NA'
Invalid operating margin for industry Banks (Regional): could not convert string to float: 'NA'
Invalid EV/Sales value for industry Brokerage & Investment Banking: could not convert string to float: 'NA'
Invalid operating margin for industry Brokerage & Investment Banking: could not convert string to float: 'NA'
Invalid EV/Sales value for industry : could not convert string to float: ''
Invalid operating margin for industry : could not convert string to float: ''
{
  "Advertising": {
    "EV/Sales": 2.75,
    "Operating Margin": 11.25
  },
  "Aerospace/Defense": {
    "EV/Sales": 2.6,
    "Operating Margin": 7.66
  },
  "Air Transport": {
    "EV/Sales": 1.02,
    "Operating Margin": 5.46
  },
  "Appa

In [7]:
for industry in market_data['industries']:
    if industry == 'Choose an Industry':
        continue
    if industry in psdata_data:
        market_data['industries'][industry]['EV/Sales'] = psdata_data[industry]['EV/Sales']
        market_data['industries'][industry]['Operating Margin'] = psdata_data[industry]['Operating Margin']
    else:
        print(f"Industry '{industry}' not found in fetched data; defaulting to None.")
        market_data['industries'][industry]['EV/Sales'] = None
        market_data['industries'][industry]['Operating Margin'] = None

print(json.dumps(market_data['industries'], indent=2))

{
  "Choose an Industry": {
    "EV/Sales": 1.1234,
    "Cost of Capital": 12.34,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 12.34,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 12.34
  },
  "Advertising": {
    "EV/Sales": 2.75,
    "Cost of Capital": 12.34,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 11.25,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 12.34
  },
  "Aerospace/Defense": {
    "EV/Sales": 2.6,
    "Cost of Capital": 12.34,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 7.66,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 12.34
  },
  "Air Transport": {
    "EV/Sales": 1.02,
    "Cost of Capital": 12.34,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 5.46,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 12.34
  },
  "Apparel": {
    "EV/Sales": 1.41,
    "Cost of Capital": 

In [8]:
wacc_url = "https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/wacc.html"

try:
    response = requests.get(wacc_url)
    response.raise_for_status()

    wacc_soup = BeautifulSoup(response.text, 'html.parser')

    all_tables = wacc_soup.find_all('table')

    wacc_table = next(table for table in all_tables if 'Advertising' in str(table))
    if not wacc_table:
        raise ValueError("Could not find the WACC table in the HTML content.")
    
except Exception as e:
    print(f"Error fetching or parsing WACC data: {e}")
    wacc_table = None

wacc_rows = wacc_table.find_all('tr')[1:]
wacc_data = {}

for row in wacc_rows:
    cols = row.find_all('td')
    industry_name = ' '.join(cols[0].get_text(strip=True).split())
    try:
        cod = float(cols[6].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid cost of debt value for industry {industry_name}: {e}")
        cod = None

    try:
        tax = float(cols[7].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid tax rate for industry {industry_name}: {e}")
        tax = None

    try:
        coc = float(cols[10].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid cost of capital for industry {industry_name}: {e}")
        coc = None

    wacc_data[industry_name] = {
        "Cost of Debt": cod,
        "Tax Rate": tax,
        "Cost of Capital": coc
    }

print(json.dumps(wacc_data, indent=2))

Invalid cost of debt value for industry : could not convert string to float: ''
Invalid tax rate for industry : could not convert string to float: ''
Invalid cost of capital for industry : could not convert string to float: ''
{
  "Advertising": {
    "Cost of Debt": 6.41,
    "Tax Rate": 7.67,
    "Cost of Capital": 9.22
  },
  "Aerospace/Defense": {
    "Cost of Debt": 5.53,
    "Tax Rate": 11.02,
    "Cost of Capital": 7.68
  },
  "Air Transport": {
    "Cost of Debt": 6.41,
    "Tax Rate": 10.15,
    "Cost of Capital": 7.29
  },
  "Apparel": {
    "Cost of Debt": 5.78,
    "Tax Rate": 8.08,
    "Cost of Capital": 7.44
  },
  "Auto & Truck": {
    "Cost of Debt": 6.41,
    "Tax Rate": 2.11,
    "Cost of Capital": 10.34
  },
  "Auto Parts": {
    "Cost of Debt": 5.78,
    "Tax Rate": 12.77,
    "Cost of Capital": 8.09
  },
  "Bank (Money Center)": {
    "Cost of Debt": 5.53,
    "Tax Rate": 18.1,
    "Cost of Capital": 5.64
  },
  "Banks (Regional)": {
    "Cost of Debt": 5.08,
    "

In [9]:
for industry in market_data['industries']:
    if industry == 'Choose an Industry':
        continue
    if industry in psdata_data:
        market_data['industries'][industry]['Cost of Debt'] = wacc_data[industry]['Cost of Debt']
        market_data['industries'][industry]['Tax Rate'] = wacc_data[industry]['Tax Rate']
        market_data['industries'][industry]['Cost of Capital'] = wacc_data[industry]['Cost of Capital']
    else:
        print(f"Industry '{industry}' not found in fetched data; defaulting to None.")
        market_data['industries'][industry]['Cost of Debt'] = None
        market_data['industries'][industry]['Tax Rate'] = None
        market_data['industries'][industry]['Cost of Capital'] = None

print(json.dumps(market_data['industries'], indent=2))

{
  "Choose an Industry": {
    "EV/Sales": 1.1234,
    "Cost of Capital": 12.34,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 12.34,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 12.34
  },
  "Advertising": {
    "EV/Sales": 2.75,
    "Cost of Capital": 9.22,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 11.25,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 7.67,
    "Cost of Debt": 6.41
  },
  "Aerospace/Defense": {
    "EV/Sales": 2.6,
    "Cost of Capital": 7.68,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 7.66,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 11.02,
    "Cost of Debt": 5.53
  },
  "Air Transport": {
    "EV/Sales": 1.02,
    "Cost of Capital": 7.29,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 5.46,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 10.15,
    "Cost of Deb

In [10]:
pbvdata_url = "https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/pbvdata.html"

try:
    response = requests.get(pbvdata_url)
    response.raise_for_status()

    pbvdata_soup = BeautifulSoup(response.text, 'html.parser')

    all_tables = pbvdata_soup.find_all('table')

    pbvdata_table = next(table for table in all_tables if 'Advertising' in str(table))
    if not pbvdata_table:
        raise ValueError("Could not find the pbvdata table in the HTML content.")
    
except Exception as e:
    print(f"Error fetching or parsing pbvdata data: {e}")
    pbvdata_table = None

pbvdata_rows = pbvdata_table.find_all('tr')[1:]
pbvdata_data = {}

for row in pbvdata_rows:
    cols = row.find_all('td')
    industry_name = ' '.join(cols[0].get_text(strip=True).split())
    try:
        roic = float(cols[5].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid ROIC value for industry {industry_name}: {e}")
        roic = None

    pbvdata_data[industry_name] = {
        "ROIC": roic
    }

print(json.dumps(pbvdata_data, indent=2))

Invalid ROIC value for industry Bank (Money Center): could not convert string to float: 'NA'
Invalid ROIC value for industry Banks (Regional): could not convert string to float: 'NA'
Invalid ROIC value for industry Brokerage & Investment Banking: could not convert string to float: 'NA'
Invalid ROIC value for industry Financial Svcs. (Non-bank & Insurance): could not convert string to float: 'NA'
Invalid ROIC value for industry : could not convert string to float: ''
{
  "Advertising": {
    "ROIC": 34.91
  },
  "Aerospace/Defense": {
    "ROIC": 14.03
  },
  "Air Transport": {
    "ROIC": 8.48
  },
  "Apparel": {
    "ROIC": 15.26
  },
  "Auto & Truck": {
    "ROIC": 3.15
  },
  "Auto Parts": {
    "ROIC": 8.66
  },
  "Bank (Money Center)": {
    "ROIC": null
  },
  "Banks (Regional)": {
    "ROIC": null
  },
  "Beverage (Alcoholic)": {
    "ROIC": 17.86
  },
  "Beverage (Soft)": {
    "ROIC": 30.61
  },
  "Broadcasting": {
    "ROIC": 12.68
  },
  "Brokerage & Investment Banking": {
 

In [11]:
for industry in market_data['industries']:
    if industry == 'Choose an Industry':
        continue
    if industry in pbvdata_data:
        market_data['industries'][industry]['ROIC'] = pbvdata_data[industry]['ROIC']
    else:
        print(f"Industry '{industry}' not found in fetched data; defaulting to None.")
        market_data['industries'][industry]['ROIC'] = None

print(json.dumps(market_data['industries'], indent=2))

{
  "Choose an Industry": {
    "EV/Sales": 1.1234,
    "Cost of Capital": 12.34,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 12.34,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 12.34
  },
  "Advertising": {
    "EV/Sales": 2.75,
    "Cost of Capital": 9.22,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 11.25,
    "Sales Cap Ratio": 12.34,
    "ROIC": 34.91,
    "Tax Rate": 7.67,
    "Cost of Debt": 6.41
  },
  "Aerospace/Defense": {
    "EV/Sales": 2.6,
    "Cost of Capital": 7.68,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 7.66,
    "Sales Cap Ratio": 12.34,
    "ROIC": 14.03,
    "Tax Rate": 11.02,
    "Cost of Debt": 5.53
  },
  "Air Transport": {
    "EV/Sales": 1.02,
    "Cost of Capital": 7.29,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 5.46,
    "Sales Cap Ratio": 12.34,
    "ROIC": 8.48,
    "Tax Rate": 10.15,
    "Cost of Debt

In [12]:
histgr_url = "https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/histgr.html"

try:
    response = requests.get(histgr_url)
    response.raise_for_status()

    histgr_soup = BeautifulSoup(response.text, 'html.parser')

    all_tables = histgr_soup.find_all('table')

    histgr_table = next(table for table in all_tables if 'Advertising' in str(table))
    if not histgr_table:
        raise ValueError("Could not find the histgr table in the HTML content.")
    
except Exception as e:
    print(f"Error fetching or parsing histgr data: {e}")
    histgr_table = None

histgr_rows = histgr_table.find_all('tr')[1:]
histgr_data = {}

for row in histgr_rows:
    cols = row.find_all('td')
    industry_name = ' '.join(cols[0].get_text(strip=True).split())
    try:
        rev_growth = float(cols[3].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid revenue growth value for industry {industry_name}: {e}")
        rev_growth = None

    histgr_data[industry_name] = {
        'Revenue Growth Rate': rev_growth
    }

print(json.dumps(histgr_data, indent=2))

Invalid revenue growth value for industry : could not convert string to float: ''
{
  "Advertising": {
    "Revenue Growth Rate": -1.62
  },
  "Aerospace/Defense": {
    "Revenue Growth Rate": 8.21
  },
  "Air Transport": {
    "Revenue Growth Rate": 2.64
  },
  "Apparel": {
    "Revenue Growth Rate": 6.97
  },
  "Auto & Truck": {
    "Revenue Growth Rate": 10.9
  },
  "Auto Parts": {
    "Revenue Growth Rate": 6.49
  },
  "Bank (Money Center)": {
    "Revenue Growth Rate": 6.0
  },
  "Banks (Regional)": {
    "Revenue Growth Rate": 6.54
  },
  "Beverage (Alcoholic)": {
    "Revenue Growth Rate": 5.09
  },
  "Beverage (Soft)": {
    "Revenue Growth Rate": 16.75
  },
  "Broadcasting": {
    "Revenue Growth Rate": 1.89
  },
  "Brokerage & Investment Banking": {
    "Revenue Growth Rate": 22.19
  },
  "Building Materials": {
    "Revenue Growth Rate": 4.06
  },
  "Business & Consumer Services": {
    "Revenue Growth Rate": 5.24
  },
  "Cable TV": {
    "Revenue Growth Rate": 24.15
  },
  

In [13]:
for industry in market_data['industries']:
    if industry == 'Choose an Industry':
        continue
    if industry in histgr_data:
        market_data['industries'][industry]['Revenue Growth Rate'] = histgr_data[industry]['Revenue Growth Rate']
    else:
        print(f"Industry '{industry}' not found in fetched data; defaulting to None.")
        market_data['industries'][industry]['Revenue Growth Rate'] = None

print(json.dumps(market_data['industries'], indent=2))

{
  "Choose an Industry": {
    "EV/Sales": 1.1234,
    "Cost of Capital": 12.34,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 12.34,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 12.34
  },
  "Advertising": {
    "EV/Sales": 2.75,
    "Cost of Capital": 9.22,
    "Beta": 12.34,
    "Revenue Growth Rate": -1.62,
    "Operating Margin": 11.25,
    "Sales Cap Ratio": 12.34,
    "ROIC": 34.91,
    "Tax Rate": 7.67,
    "Cost of Debt": 6.41
  },
  "Aerospace/Defense": {
    "EV/Sales": 2.6,
    "Cost of Capital": 7.68,
    "Beta": 12.34,
    "Revenue Growth Rate": 8.21,
    "Operating Margin": 7.66,
    "Sales Cap Ratio": 12.34,
    "ROIC": 14.03,
    "Tax Rate": 11.02,
    "Cost of Debt": 5.53
  },
  "Air Transport": {
    "EV/Sales": 1.02,
    "Cost of Capital": 7.29,
    "Beta": 12.34,
    "Revenue Growth Rate": 2.64,
    "Operating Margin": 5.46,
    "Sales Cap Ratio": 12.34,
    "ROIC": 8.48,
    "Tax Rate": 10.15,
    "Cost of Debt":

In [14]:
capex_url = "https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/capex.html"

try:
    response = requests.get(capex_url)
    response.raise_for_status()

    capex_soup = BeautifulSoup(response.text, 'html.parser')

    all_tables = capex_soup.find_all('table')

    capex_table = next(table for table in all_tables if 'Advertising' in str(table))
    if not capex_table:
        raise ValueError("Could not find the capex table in the HTML content.")
    
except Exception as e:
    print(f"Error fetching or parsing capex data: {e}")
    capex_table = None

capex_rows = capex_table.find_all('tr')[1:]
capex_data = {}

for row in capex_rows:
    cols = row.find_all('td')
    industry_name = ' '.join(cols[0].get_text(strip=True).split())
    try:
        sales_cap = float(cols[9].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid sales cap value for industry {industry_name}: {e}")
        sales_cap = None

    capex_data[industry_name] = {
        'Sales Cap Ratio': sales_cap
    }

print(json.dumps(capex_data, indent=2))

Invalid sales cap value for industry Bank (Money Center): could not convert string to float: 'NA'
Invalid sales cap value for industry Banks (Regional): could not convert string to float: 'NA'
Invalid sales cap value for industry Brokerage & Investment Banking: could not convert string to float: 'NA'
Invalid sales cap value for industry : could not convert string to float: ''
{
  "Advertising": {
    "Sales Cap Ratio": 4.19
  },
  "Aerospace/Defense": {
    "Sales Cap Ratio": 2.62
  },
  "Air Transport": {
    "Sales Cap Ratio": 1.75
  },
  "Apparel": {
    "Sales Cap Ratio": 1.73
  },
  "Auto & Truck": {
    "Sales Cap Ratio": 1.15
  },
  "Auto Parts": {
    "Sales Cap Ratio": 2.43
  },
  "Bank (Money Center)": {
    "Sales Cap Ratio": null
  },
  "Banks (Regional)": {
    "Sales Cap Ratio": null
  },
  "Beverage (Alcoholic)": {
    "Sales Cap Ratio": 0.85
  },
  "Beverage (Soft)": {
    "Sales Cap Ratio": 1.67
  },
  "Broadcasting": {
    "Sales Cap Ratio": 1.14
  },
  "Brokerage & I

In [15]:
for industry in market_data['industries']:
    if industry == 'Choose an Industry':
        continue
    if industry in capex_data:
        market_data['industries'][industry]['Sales Cap Ratio'] = capex_data[industry]['Sales Cap Ratio']
    else:
        print(f"Industry '{industry}' not found in fetched data; defaulting to None.")
        market_data['industries'][industry]['Sales Cap Ratio'] = None

print(json.dumps(market_data['industries'], indent=2))

{
  "Choose an Industry": {
    "EV/Sales": 1.1234,
    "Cost of Capital": 12.34,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 12.34,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 12.34
  },
  "Advertising": {
    "EV/Sales": 2.75,
    "Cost of Capital": 9.22,
    "Beta": 12.34,
    "Revenue Growth Rate": -1.62,
    "Operating Margin": 11.25,
    "Sales Cap Ratio": 4.19,
    "ROIC": 34.91,
    "Tax Rate": 7.67,
    "Cost of Debt": 6.41
  },
  "Aerospace/Defense": {
    "EV/Sales": 2.6,
    "Cost of Capital": 7.68,
    "Beta": 12.34,
    "Revenue Growth Rate": 8.21,
    "Operating Margin": 7.66,
    "Sales Cap Ratio": 2.62,
    "ROIC": 14.03,
    "Tax Rate": 11.02,
    "Cost of Debt": 5.53
  },
  "Air Transport": {
    "EV/Sales": 1.02,
    "Cost of Capital": 7.29,
    "Beta": 12.34,
    "Revenue Growth Rate": 2.64,
    "Operating Margin": 5.46,
    "Sales Cap Ratio": 1.75,
    "ROIC": 8.48,
    "Tax Rate": 10.15,
    "Cost of Debt": 6.

In [16]:
beta_url = "https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/Betas.html"

try:
    response = requests.get(beta_url)
    response.raise_for_status()

    beta_soup = BeautifulSoup(response.text, 'html.parser')

    all_tables = beta_soup.find_all('table')

    beta_table = next(table for table in all_tables if 'Advertising' in str(table))
    if not beta_table:
        raise ValueError("Could not find the beta table in the HTML content.")
    
except Exception as e:
    print(f"Error fetching or parsing beta data: {e}")
    beta_table = None

beta_rows = beta_table.find_all('tr')[1:]
beta_data = {}

for row in beta_rows:
    cols = row.find_all('td')
    industry_name = ' '.join(cols[0].get_text(strip=True).split())
    try:
        beta = float(cols[5].get_text(strip=True).replace('%', ''))
    except Exception as e:
        print(f"Invalid beta value for industry {industry_name}: {e}")
        beta = None

    beta_data[industry_name] = {
        'Beta': beta
    }

print(json.dumps(beta_data, indent=2))

Invalid beta value for industry : could not convert string to float: ''
{
  "Advertising": {
    "Beta": 1.12
  },
  "Aerospace/Defense": {
    "Beta": 0.77
  },
  "Air Transport": {
    "Beta": 0.69
  },
  "Apparel": {
    "Beta": 0.74
  },
  "Auto & Truck": {
    "Beta": 1.39
  },
  "Auto Parts": {
    "Beta": 0.91
  },
  "Bank (Money Center)": {
    "Beta": 0.37
  },
  "Banks (Regional)": {
    "Beta": 0.36
  },
  "Beverage (Alcoholic)": {
    "Beta": 0.5
  },
  "Beverage (Soft)": {
    "Beta": 0.5
  },
  "Broadcasting": {
    "Beta": 0.43
  },
  "Brokerage & Investment Banking": {
    "Beta": 0.4
  },
  "Building Materials": {
    "Beta": 1.19
  },
  "Business & Consumer Services": {
    "Beta": 0.89
  },
  "Cable TV": {
    "Beta": 0.49
  },
  "Chemical (Basic)": {
    "Beta": 0.8
  },
  "Chemical (Diversified)": {
    "Beta": 0.54
  },
  "Chemical (Specialty)": {
    "Beta": 0.76
  },
  "Coal & Related Energy": {
    "Beta": 1.1
  },
  "Computer Services": {
    "Beta": 1.03
  },

In [17]:
for industry in market_data['industries']:
    if industry == 'Choose an Industry':
        continue
    if industry in beta_data:
        market_data['industries'][industry]['Beta'] = beta_data[industry]['Beta']
    else:
        print(f"Industry '{industry}' not found in fetched data; defaulting to None.")
        market_data['industries'][industry]['Beta'] = None
print(json.dumps(market_data['industries'], indent=2))

{
  "Choose an Industry": {
    "EV/Sales": 1.1234,
    "Cost of Capital": 12.34,
    "Beta": 12.34,
    "Revenue Growth Rate": 12.34,
    "Operating Margin": 12.34,
    "Sales Cap Ratio": 12.34,
    "ROIC": 12.34,
    "Tax Rate": 12.34
  },
  "Advertising": {
    "EV/Sales": 2.75,
    "Cost of Capital": 9.22,
    "Beta": 1.12,
    "Revenue Growth Rate": -1.62,
    "Operating Margin": 11.25,
    "Sales Cap Ratio": 4.19,
    "ROIC": 34.91,
    "Tax Rate": 7.67,
    "Cost of Debt": 6.41
  },
  "Aerospace/Defense": {
    "EV/Sales": 2.6,
    "Cost of Capital": 7.68,
    "Beta": 0.77,
    "Revenue Growth Rate": 8.21,
    "Operating Margin": 7.66,
    "Sales Cap Ratio": 2.62,
    "ROIC": 14.03,
    "Tax Rate": 11.02,
    "Cost of Debt": 5.53
  },
  "Air Transport": {
    "EV/Sales": 1.02,
    "Cost of Capital": 7.29,
    "Beta": 0.69,
    "Revenue Growth Rate": 2.64,
    "Operating Margin": 5.46,
    "Sales Cap Ratio": 1.75,
    "ROIC": 8.48,
    "Tax Rate": 10.15,
    "Cost of Debt": 6.41


In [18]:
ratings_url = "https://pages.stern.nyu.edu/~adamodar/New_Home_Page/datafile/ratings.html"

try:
    response = requests.get(ratings_url)
    response.raise_for_status()

    ratings_soup = BeautifulSoup(response.text, 'html.parser')

    all_tables = ratings_soup.find_all('table')

    ratings_table = all_tables[0]

    if not ratings_table:
        raise ValueError("Could not find the ratings table in the HTML content.")
    
except Exception as e:
    print(f"Error fetching or parsing ratings data: {e}")
    all_tables = None

ratings_rows = ratings_table.find_all('tr')[1:]
pruned_ratings_rows = []
ratings_data = {}

for row in ratings_rows:
    cols = row.find_all('td')
    if cols[2].getText(strip=True):
        if cols[2].getText(strip=True) == "Rating is":
            continue
        pruned_ratings_rows.append([
            cols[0].get_text(strip=True),
            cols[1].get_text(strip=True),
            cols[2].get_text(strip=True),
            cols[3].get_text(strip=True).replace('%', '')
        ])

large_ratings = pruned_ratings_rows[:15]
small_ratings = pruned_ratings_rows[15:]

for [gt, lt, rating, spread] in large_ratings:
    ratings_data[rating] = {}
    try:
        ratings_data[rating]['gt_safe'] = float(gt)
        ratings_data[rating]['lt_safe'] = float(lt)
        ratings_data[rating]['Spread'] = float(spread)
    except Exception as e:
        print(f"Invalid rating bounds for rating {rating}: {e}")
        ratings_data[rating]['gt_safe'] = None
        ratings_data[rating]['lt_safe'] = None
        ratings_data[rating]['Spread'] = None

for [gt, lt, rating, spread] in small_ratings:
    try:
        ratings_data[rating]['gt_risk'] = float(gt)
        ratings_data[rating]['lt_risk'] = float(lt)
    except Exception as e:
        print(f"Invalid rating bounds for rating {rating}: {e}")
        ratings_data[rating]['gt_risk'] = None
        ratings_data[rating]['lt_risk'] = None

print(json.dumps(ratings_data, indent=2))

{
  "D2/D": {
    "gt_safe": -100000.0,
    "lt_safe": 0.199999,
    "Spread": 19.0,
    "gt_risk": -100000.0,
    "lt_risk": 0.499999
  },
  "C2/C": {
    "gt_safe": 0.2,
    "lt_safe": 0.649999,
    "Spread": 15.5,
    "gt_risk": 0.5,
    "lt_risk": 0.799999
  },
  "Ca2/CC": {
    "gt_safe": 0.65,
    "lt_safe": 0.799999,
    "Spread": 10.1,
    "gt_risk": 0.8,
    "lt_risk": 1.249999
  },
  "Caa/CCC": {
    "gt_safe": 0.8,
    "lt_safe": 1.249999,
    "Spread": 7.28,
    "gt_risk": 1.25,
    "lt_risk": 1.499999
  },
  "B3/B-": {
    "gt_safe": 1.25,
    "lt_safe": 1.499999,
    "Spread": 4.42,
    "gt_risk": 1.5,
    "lt_risk": 1.999999
  },
  "B2/B": {
    "gt_safe": 1.5,
    "lt_safe": 1.749999,
    "Spread": 3.0,
    "gt_risk": 2.0,
    "lt_risk": 2.499999
  },
  "B1/B+": {
    "gt_safe": 1.75,
    "lt_safe": 1.999999,
    "Spread": 2.61,
    "gt_risk": 2.5,
    "lt_risk": 2.999999
  },
  "Ba2/BB": {
    "gt_safe": 2.0,
    "lt_safe": 2.2499999,
    "Spread": 1.83,
    "gt_risk":

In [19]:
for rating in market_data['credit_ratings']:
    if rating == 'Select a Credit Rating':
        continue
    if rating in ratings_data:
        market_data['credit_ratings'][rating]['gt_safe'] = ratings_data[rating]['gt_safe']
        market_data['credit_ratings'][rating]['lt_safe'] = ratings_data[rating]['lt_safe']
        market_data['credit_ratings'][rating]['gt_risk'] = ratings_data[rating]['gt_risk']
        market_data['credit_ratings'][rating]['lt_risk'] = ratings_data[rating]['lt_risk']
        market_data['credit_ratings'][rating]['Spread'] = ratings_data[rating]['Spread']
    else:
        print(f"Rating '{rating}' not found in fetched data; defaulting to None.")
        market_data['credit_ratings'][rating]['gt_safe'] = None
        market_data['credit_ratings'][rating]['lt_safe'] = None
        market_data['credit_ratings'][rating]['gt_risk'] = None
        market_data['credit_ratings'][rating]['lt_risk'] = None
        market_data['credit_ratings'][rating]['Spread'] = None

print(json.dumps(market_data['credit_ratings'], indent=2))

{
  "Select a Credit Rating": {
    "Spread": 12.34,
    "gt_safe": 100001,
    "gt_risk": 100001,
    "lt_safe": -100001,
    "lt_risk": -100001
  },
  "Aaa/AAA": {
    "Spread": 0.45,
    "gt_safe": 8.5,
    "gt_risk": 12.5,
    "lt_safe": 100000.0,
    "lt_risk": 100000.0
  },
  "Aa2/AA": {
    "Spread": 0.6,
    "gt_safe": 6.5,
    "gt_risk": 9.5,
    "lt_safe": 8.499999,
    "lt_risk": 12.499999
  },
  "A1/A+": {
    "Spread": 0.77,
    "gt_safe": 5.5,
    "gt_risk": 7.5,
    "lt_safe": 6.499999,
    "lt_risk": 9.499999
  },
  "A2/A": {
    "Spread": 0.85,
    "gt_safe": 4.25,
    "gt_risk": 6.0,
    "lt_safe": 5.499999,
    "lt_risk": 7.499999
  },
  "A3/A-": {
    "Spread": 0.95,
    "gt_safe": 3.0,
    "gt_risk": 4.5,
    "lt_safe": 4.249999,
    "lt_risk": 5.999999
  },
  "Baa2/BBB": {
    "Spread": 1.2,
    "gt_safe": 2.5,
    "gt_risk": 4.0,
    "lt_safe": 2.999999,
    "lt_risk": 4.499999
  },
  "Ba1/BB+": {
    "Spread": 1.55,
    "gt_safe": 2.25,
    "gt_risk": 3.5,
    "

In [20]:
# last item, regional erp seems to only be saved on an xlsx file
# will parse this data using pandas instead
ctryprem_xlsx_url = "https://www.stern.nyu.edu/~adamodar/pc/datasets/ctryprem.xlsx"
try:
    response = requests.get(ctryprem_xlsx_url)
    response.raise_for_status()

    ctryprem_xlsx_file = BytesIO(response.content)

    df = pd.read_excel(ctryprem_xlsx_file, sheet_name='Regional Weighted Averages')

    emea_erp = df[df.iloc[:, 0] == 'EMEA'].iloc[0, 1]

    ctryprem_df = df.iloc[169:179, :2]
    ctryprem_df.columns = ['Region', 'ERP']
    ctryprem_df = ctryprem_df.set_index('Region')

except Exception as e:
    print(f"Error fetching or parsing country premium xlsx data: {e}")

print('EMEA ERP: ', emea_erp)
print(ctryprem_df)

EMEA ERP:  0.0591478631288238
                                ERP
Region                             
Africa                     0.119493
Asia                       0.057223
Australia & New Zealand    0.042341
Caribbean                  0.117177
Central and South America  0.084628
Eastern Europe             0.075777
Middle East                0.062367
North America              0.044465
Western Europe             0.052665
Global                     0.056257


In [21]:
for region in market_data['regions']:
    if region == 'Select a Region' or region == 'EMEA':
        continue
    if region in ctryprem_df.index:
        market_data['regions'][region]['ERP'] = ctryprem_df.loc[region, 'ERP'] * 100
    else:
        print(f"Region '{region}' not found in fetched data; defaulting to None.")
        market_data['regions'][region]['ERP'] = None
    
market_data['regions']['EMEA']['ERP'] = emea_erp * 100

print('Manually adding Rest of World as Global ERP.')
market_data['regions']['Rest of World']['ERP'] = ctryprem_df.loc['Global', 'ERP'] * 100

print(json.dumps(market_data['regions'], indent=2))

Region 'Rest of World' not found in fetched data; defaulting to None.
Manually adding Rest of World as Global ERP.
{
  "Select a Region": {
    "ERP": 12.34
  },
  "Africa": {
    "ERP": 11.949308050536839
  },
  "Asia": {
    "ERP": 5.722323564845226
  },
  "Australia & New Zealand": {
    "ERP": 4.234082173200422
  },
  "Caribbean": {
    "ERP": 11.7176823349771
  },
  "Central and South America": {
    "ERP": 8.462849620618984
  },
  "Eastern Europe": {
    "ERP": 7.577709101845076
  },
  "Middle East": {
    "ERP": 6.2366864403681905
  },
  "North America": {
    "ERP": 4.446474878479405
  },
  "Western Europe": {
    "ERP": 5.2665467678857745
  },
  "EMEA": {
    "ERP": 5.91478631288238
  },
  "Rest of World": {
    "ERP": 5.625707330046292
  }
}


In [22]:
# Market data should be fully populated now, write to file in FrontEnd
try:
    with open('../FrontEnd/src/utils/marketData.json', 'w') as f:
        json.dump(market_data, f, indent=2)
        print("Market data successfully written to marketData.json")
except Exception as e:
    print(f"Error writing market data to file: {e}")

Market data successfully written to marketData.json
